> ### ▶ Running this notebook in Google Colab (recommended)> 1. Go to [colab.research.google.com](https://colab.research.google.com)> 2. Click **File → Upload notebook**> 3. Select this `.ipynb` file from your downloaded course ZIP> 4. Click **Runtime → Run all**>> Alternatively: upload the whole ZIP to **Google Drive**, then double-click any notebook — it opens in Colab automatically.>> _No installation required. All you need is a free Google account._

## 2.1 — Setup

In [ ]:
import sysIN_COLAB = 'google.colab' in sys.modulesif IN_COLAB:    import subprocess    subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl', '--quiet'])import pandas as pdimport numpy as npimport matplotlib.pyplot as pltprint("✅ Ready")

---## 2.2 — VLOOKUP → `merge()`VLOOKUP is one of the most-used Excel functions in finance. It has two problems: it breaks when columns are inserted, and it's slow on large datasets.`pd.merge()` is the pandas equivalent — think of it as a proper database JOIN. It's faster, safer, and more readable.**Excel:** `=VLOOKUP(A2, Rates!$A:$B, 2, FALSE)`**Python:** `pd.merge(df, rates, on='currency', how='left')`

In [ ]:
# Simulate a revenue table with currency codesrevenue = pd.DataFrame({    'Deal':     ['A001', 'A002', 'A003', 'A004', 'A005'],    'Currency': ['USD',  'EUR',  'GBP',  'USD',  'EUR'],    'Amount':   [500000, 320000, 180000, 410000, 275000]})# Simulate an FX rates lookup table (like a second sheet in Excel)fx_rates = pd.DataFrame({    'Currency': ['USD', 'EUR', 'GBP'],    'Rate_to_GBP': [0.79, 0.86, 1.00]})print("Revenue table:")print(revenue)print()print("FX rates table:")print(fx_rates)

In [ ]:
# VLOOKUP equivalent — merge on the common key columnmerged = pd.merge(revenue, fx_rates, on='Currency', how='left')# Calculate GBP equivalentmerged['Amount_GBP'] = (merged['Amount'] * merged['Rate_to_GBP']).round(0)print("After merge (VLOOKUP equivalent):")merged

**Key differences from VLOOKUP:**- Works on any column, not just the leftmost- `how='left'` keeps all rows from the left table (like VLOOKUP with IFERROR)- `how='inner'` only keeps matched rows- Never breaks when you insert or delete columns---## 2.3 — SUMIF / COUNTIF → `groupby()`SUMIF and COUNTIF aggregate data based on a condition. In pandas this is `groupby()` — and it handles multiple conditions and multiple aggregations at once.

In [ ]:
# Simulate a transactions tabletransactions = pd.DataFrame({    'Month':      ['Jan','Jan','Jan','Feb','Feb','Feb','Mar','Mar','Mar'],    'Region':     ['North','South','North','South','North','South','North','South','North'],    'Product':    ['A','B','A','A','B','B','A','A','B'],    'Revenue':    [120, 85, 95, 110, 140, 75, 130, 88, 160],    'Units':      [12,  9,  10, 11,  14,  8,  13,  9,  16]})transactions

In [ ]:
# SUMIF equivalent: total revenue by region# Excel: =SUMIF(B:B, "North", D:D)by_region = transactions.groupby('Region')['Revenue'].sum()print("Revenue by Region (SUMIF equivalent):")print(by_region)print()# COUNTIF equivalent: count transactions by monthby_month_count = transactions.groupby('Month')['Revenue'].count()print("Transaction count by Month (COUNTIF equivalent):")print(by_month_count)

In [ ]:
# Multiple aggregations at once — impossible cleanly in Excel without helper columnssummary = transactions.groupby(['Month', 'Region']).agg(    Total_Revenue=('Revenue', 'sum'),    Total_Units=('Units', 'sum'),    Avg_Deal_Size=('Revenue', 'mean'),    Num_Deals=('Revenue', 'count')).round(1).reset_index()summary

---## 2.4 — Pivot Tables → `pivot_table()`Pandas has a direct `pivot_table()` function that maps almost exactly to Excel's Insert → PivotTable.

In [ ]:
# Revenue pivot: Months as columns, Regions as rowspivot = pd.pivot_table(    transactions,    values='Revenue',    index='Region',      # Row labels    columns='Month',     # Column labels    aggfunc='sum',       # Equivalent to "Sum of Revenue" in Excel    margins=True,        # Adds a Grand Total row/column    margins_name='Total')print("Pivot Table — Revenue by Region and Month:")pivot

In [ ]:
# You can pivot on multiple values at oncepivot_multi = pd.pivot_table(    transactions,    values=['Revenue', 'Units'],    index='Region',    columns='Month',    aggfunc='sum')pivot_multi

---## 2.5 — Date HandlingDates in Excel are secretly just integers (days since 1 Jan 1900). This causes endless problems with formatting, sorting, and calculations. Pandas uses proper datetime objects.This matters enormously in finance: fiscal quarters, working days, month-end dates, YoY comparisons.

In [ ]:
# Create a date-indexed time series — common in finance reportingdates = pd.date_range(start='2024-01-01', end='2024-12-31', freq='ME')  # Month-end datesfinancials = pd.DataFrame({    'Date':    dates,    'Revenue': [120,135,128,142,155,161,158,149,163,171,182,195],    'Costs':   [ 78, 85, 82, 91, 98,102,100, 95,103,108,115,122]})financials['Date'] = pd.to_datetime(financials['Date'])financials.set_index('Date', inplace=True)financials['Profit'] = financials['Revenue'] - financials['Costs']financials

In [ ]:
# Extract date components — replaces MONTH(), YEAR(), QUARTER() in Excelfinancials['Month']   = financials.index.monthfinancials['Quarter'] = financials.index.quarterfinancials['Year']    = financials.index.year# Quarterly rollupquarterly = financials.groupby('Quarter')[['Revenue','Costs','Profit']].sum()quarterly['Margin %'] = (quarterly['Profit'] / quarterly['Revenue'] * 100).round(1)print("Quarterly Summary:")quarterly

In [ ]:
# Month-over-month growth — replaces complex Excel offset formulasfinancials['Revenue_MoM%'] = financials['Revenue'].pct_change() * 100financials['Revenue_MoM%'] = financials['Revenue_MoM%'].round(1)# Rolling 3-month average — replaces AVERAGE(OFFSET(...)) in Excelfinancials['Revenue_3M_Avg'] = financials['Revenue'].rolling(window=3).mean().round(1)financials[['Revenue','Revenue_MoM%','Revenue_3M_Avg']].dropna()

---## 2.6 — Conditional Logic → `np.where()` and `apply()`Excel's IF, IFS, and nested IFs have clean pandas equivalents.

In [ ]:
# Simple IF equivalent: np.where(condition, value_if_true, value_if_false)# Excel: =IF(D2>150, "High", "Normal")financials['Revenue_Band'] = np.where(financials['Revenue'] > 150, 'High', 'Normal')# Nested IFS equivalent using np.selectconditions = [    financials['Revenue'] >= 175,    financials['Revenue'] >= 145,    financials['Revenue'] >= 125,]choices = ['Tier 1', 'Tier 2', 'Tier 3']financials['Tier'] = np.select(conditions, choices, default='Below Target')financials[['Revenue', 'Revenue_Band', 'Tier']]

---## 2.7 — Key Takeaways- `merge()` replaces VLOOKUP — it's safer, faster, and never breaks on column inserts- `groupby().agg()` replaces SUMIF/COUNTIF — and handles multiple conditions and aggregations at once- `pivot_table()` maps directly to Excel pivot tables, with the same row/column/value/aggfunc logic- Pandas datetime is far more powerful than Excel dates — month-end ranges, rolling averages, and MoM growth are one-liners- `np.where()` and `np.select()` replace IF and nested IFS cleanly---## Up Next: Module 3 — P&L ModellingWe'll build a full 3-year P&L model with dynamic assumptions, scenario analysis, and professional charts.